In [5]:
import os
import ants
import pickle
import numpy as np
import nibabel as nib
import cortex

In [ ]:
# file = "/home/zachkaras/fmri/fmri_model/analysis/fir/midprocess/105/code_formatted_keystrokes.pkl"
# with open(file, 'rb') as f:
#     keys = pickle.load(f)

In [6]:
atlas_base_path = "/home/zachkaras/fmri/fmri_model/analysis/pipeline/atlases"

# read in 2d mni mask
mask = nib.load(f"{atlas_base_path}/MNI152_T1_2mm_brain_mask.nii.gz")
og_shape = mask.shape
mask = mask.get_fdata().flatten()
brain_idx = np.where(mask>0)[0]

atlas = nib.load(f"{atlas_base_path}/Schaefer2018_400Parcels_7Networks_order_FSLMNI152_2mm.nii.gz")
atlas_vec = atlas.get_fdata().flatten()
atlas_only_brain = atlas_vec[brain_idx]
cortex_vx = np.where(atlas_only_brain != 0)[0]

# Making empty templates to save output
empty_schaefer = np.zeros(atlas_only_brain.shape)
empty_mni = np.zeros(atlas_vec.shape)
    
# def create_histogram(voxcorrs):
#     f = plt.figure(figsize=(8,8))
#     ax = f.add_subplot(1,1,1)
#     ax.hist(voxcorrs, 100) # histogram correlations with 100 bins
#     ax.set_xlabel("Correlation")
#     ax.set_ylabel("Num. voxels");
#     plt.savefig() # TODO

def convert_to_nifti(values):
    # working backwards to save correlation values as voxels in MNI space
    empty_schaefer[cortex_vx] = values
    empty_mni[brain_idx] = empty_schaefer
    result_brain = np.reshape(empty_mni, og_shape)

    # Saving results
    nifti_result = nib.Nifti1Image(result_brain, affine=atlas.affine, header=atlas.header)
    nib.save(nifti_result, "test_plotting.nii.gz")
    return result_brain, nifti_result


# Parse the ITK transform file
def read_itk_affine(filename):
    """Read ITK affine transform and convert to 4x4 matrix"""
    with open(filename, 'r') as f:
        lines = f.readlines()
    
    # Extract parameters and fixed parameters
    for line in lines:
        if line.startswith('Parameters:'):
            params = np.array([float(x) for x in line.split(':')[1].split()])
        elif line.startswith('FixedParameters:'):
            fixed = np.array([float(x) for x in line.split(':')[1].split()])
    
    # ITK affine format: first 9 params are the 3x3 rotation/scale matrix
    # last 3 params are the translation
    matrix = params[:9].reshape(3, 3)
    translation = params[9:12]
    center = fixed  # rotation center
    
    # Convert to 4x4 homogeneous matrix
    # The ITK transform is: y = Matrix * (x - center) + translation + center
    # Which simplifies to: y = Matrix * x + (translation + center - Matrix * center)
    offset = translation + center - matrix.dot(center)
    
    affine = np.eye(4)
    affine[:3, :3] = matrix
    affine[:3, 3] = offset
    
    return affine

In [ ]:
filepath = "/storage1/fmri_model_data/ridge_regression_pca_models/111/codegemma_7b-layer_28-code-correlations.pkl"
# filepath = "/storage1/fmri_model_data/ridge_regression_pca_models/122/deepseek_2b-layer_18-code-correlations.pkl"
with open(filepath, 'rb') as f:
    data = pickle.load(f)

npy_brain, nifti_brain = convert_to_nifti(data)

In [18]:
nii = nib.load("test_plotting.nii.gz")
data = nii.get_fdata()
data = data.transpose(2,1,0)
# data = np.asanyarray(nii.dataobj).transpose(2, 1, 0)  # (Z,Y,X)


In [ ]:
subject = 'fsaverage'
xfmname = 'mni2func'

# from cortex import xfm

In [ ]:
mni_ref = "/home/zachkaras/fmri/fmri_model/analysis/pipeline/atlases/MNI152_T1_2mm_brain.nii.gz"
anat = "/storage1/fmri_model_data/midprocess/111/Warped.nii.gz"
# anat = "/storage1/fmri_model_data/midprocess/122/Warped.nii.gz"

# os.system(f"flirt -in {anat} -ref {mni} -out test_registered -omat affine.mat")

In [ ]:
# TODO - transform from MNI to fsaverage 

In [ ]:
ref = "/storage1/fmri_model_data/midprocess/111/mean_func.nii.gz"
ref = "/home/zachkaras/fsl/data/standard/MNI152_T1_2mm_brain.nii.gz"
# cortex.align.automatic(subject='sub-111', xfmname='plots', reference=ref)
cortex.align.automatic(subject='mni_2mm', xfmname='plots', reference=ref)

In [ ]:
cortex.align.manual(subject='sub-111', xfmname="plots", use_fs_bbr=True)

In [ ]:
# cortex.freesurfer.import_subj(freesurfer_subject='sub-111', pycortex_subject='sub-111')
cortex.freesurfer.import_subj(freesurfer_subject='mni_2mm', pycortex_subject='mni_2mm')


In [ ]:
# affine_file = "/storage1/fmri_model_data/midprocess/111/anat2mni.txt"
# affine_file = "/storage1/fmri_model_data/midprocess/111/anat2mni_affine.txt"
affine_file = "/home/zachkaras/fmri/fmri_model/analysis/plotting/test_affine.txt"
affine = read_itk_affine(affine_file)

In [ ]:
cortex.db.save_xfm(subject='sub-111', name='test', xfm=affine, reference=mni_ref)

In [ ]:
transform = cortex.xfm.Transform(affine, reference=mni_ref)

In [ ]:
aff = "/storage1/fmri_model_data/midprocess/111/anat_2_mni_raw_affine.txt"
test = cortex.mni.compute_mni_transform(subject='sub-111', xfm=affine_file, template=mni_ref)
print(test)

In [ ]:
cortex.db.save_xfm(subject='sub-111', name='test', xfm=transform, reference=mni_ref)

In [ ]:
nonlin = "/storage1/fmri_model_data/midprocess/111/1Warp.nii.gz"
warp = nib.load(nonlin)
# transform.addWarp(warp)

In [ ]:
test = cortex.mni.compute_mni_transform(subject='sub-111', xfm='test', template=mni_ref)

In [ ]:
affine

In [ ]:
# cortex.db.save_xfm(subject='fsaverage', name='mni2py4', xfm=affine, reference="/home/zachkaras/freesurfer/subjects/fsaverage/mri/brain.nii")
cortex.db.save_xfm(subject='sub-111', name='mni2py', xfm=affine, reference=mni_ref)


In [ ]:
test = cortex.mosaic(data)

In [21]:
vol = cortex.Volume(
    data,
    subject='fsaverage',
    xfmname='mni2py2',
)

In [ ]:
from cortex.quickflat import make_figure

fig = make_figure(
    vol,
    with_rois=False,
    with_labels=False,
    with_sulci=False,
    with_borders=True,
    thick=5
)

In [22]:
# cortex.webshow(vol)
cortex.webshow(vol)
# cortex.quickshow(vol, with_labels=False)

Started server on port 22548


<JS: window.viewer>

In [ ]:
# import scipy.io as sio

mat = np.loadtxt("affine.mat")
func = "test_plotting.nii.gz"

test = xfm.Transform.from_fsl(xfm=xfm, anat_nii=anat, func_nii=func)
# test = xfm.Transform()
# need xfm and reference

In [ ]:
mni.compute_mni_transform(
    subject='test',
    xfm = 'identity',
)

In [ ]:
surf = mni.transform_mni_to_subject(
    data = nii.get_fdata(),
    affine = nii.affine,
    subject = 'fsaverage',
    target = 'surface'
)

In [ ]:
dir(cortex.surfs)

In [ ]:
mni_data = nifti_brain.get_fdata()
mni_affine = nifti_brain.affine

In [ ]:
# import cortex
from cortex import mni

# subject: Any,
#     xfm: Any,
#     volarray: Any,
#     func_to_mni: Any,
#     template: str = default_template

vol = mni.transform_mni_to_subject(
    data=mni_data,
    subject='fsaverage',
    affine=mni_affine
)

In [ ]:
# Plot mosaic of correlations
from matplotlib.pyplot import figure, cm
import matplotlib.pyplot as plt
# corrvolume = np.zeros(mask.shape)
# corrvolume[mask>0] = voxcorrs

voxel_vol = cortex.Volume(npy_brain, "test", "fullhead")

# Then we have to get a mapper from voxels to vertices for this transform
mapper = cortex.get_mapper("test", "fullhead", 'line_nearest', recache=True)

# Just pass the voxel data through the mapper to get vertex data
vertex_map = mapper(voxel_vol)

# You can plot both as you would normally plot Volume and Vertex data
cortex.quickshow(voxel_vol)
plt.show()
cortex.quickshow(vertex_map)
plt.show()

# f = figure(figsize=(10,10))
# cortex.mosaic(npy_brain, vmin=0, vmax=0.5, cmap=cm.hot);

# Create a cortex.Volume object from the NumPy array
# You can specify colormap, vmin, vmax, and a description
# volume_data = cortex.Volume(npy_brain, "test", "test",
#                             cmap='viridis', vmin=0, vmax=1,
#                             description='Example NumPy array plot')

# # Display the volume data using quickshow
# cortex.quickshow(volume_data)

# You can also save the visualization as a web page
# cortex.webshow(volume_data, filename='numpy_array_plot.html')

In [ ]:
import cortex
# nifti_img = nib.load("/storage1/fmri_model_data/ridge_regression_pca_models/111/codegemma_7b-layer_28-prose-correlations.pkl")
# data = nifti_img.get_fdata()
vol_data = cortex.Volume(nifti)

In [ ]:
import cortex
import nibabel as nib
# Load the NIFTI file: Use nibabel to load your NIFTI image.
# Python

nifti_img = nib.load('your_nifti_file.nii.gz')
data = nifti_img.get_fdata()
# Create a cortex.Volume object: This object encapsulates your volumetric data and its spatial information. You need to provide the data, a subject ID, and a transform name (which defines how the volume aligns with the surface).
# Python

# Assuming 'subject_id' and 'transform_name' are already defined in your pycortex database
# For example, if you have a subject 'S1' and a transform 'func_to_anat'
volume_data = cortex.Volume(data, 'subject_id', 'transform_name')
# Plot the data on the flatmap: Use cortex.quickflat.make_figure to generate a flatmap visualization of your data.
# Python

cortex.quickflat.make_figure(volume_data, with_colorbar=True)